In [ ]:
import pandas as pd
import numpy as np

path = "../../data/ratings-20260117-20260217.parquet"

df = pd.read_parquet(path, engine="fastparquet")

print("Rows:", len(df))
print("Columns:", df.shape[1])
df.head(3)

Rows: 6499635
Columns: 9


,noteId,ratedOnTweetId,raterParticipantId,noteAuthorParticipantId,postCreatedAt,noteCreatedAt,ratingCreatedAt,helpfulnessLevel,fromNotification
0,1357294499052085249,1357269755112148993,FE4BA8F630E66459C9F2EEBDDF69C4B22D222AD55E9013...,FB8A5B50F04AAB7CCEA99B4094EAA1AD8E49E3503EF18E...,2021-02-04 10:08:21.128,2021-02-04 11:46:40.544,2026-02-01 14:36:13.314,NOT_HELPFUL,False
1,1357297935407603713,1357269755112148993,FE4BA8F630E66459C9F2EEBDDF69C4B22D222AD55E9013...,B9BEE8138A8A67538391BC79C1E9DFF614FCEE2DD089CB...,2021-02-04 10:08:21.128,2021-02-04 12:00:19.835,2026-02-01 14:36:04.345,NOT_HELPFUL,False
2,1358604958673719296,1358464064322691078,456D1C422BA1A09690B986C8A1DBE7627D10462AF965A0...,FAB1117EE31D648BA7C11062F626AF7B6908D551B4C9C8...,2021-02-07 17:14:06.633,2021-02-08 02:33:58.465,2026-02-09 08:56:34.712,NOT_HELPFUL,False


In [15]:
df.columns

Index(['noteId', 'ratedOnTweetId', 'raterParticipantId',
       'noteAuthorParticipantId', 'postCreatedAt', 'noteCreatedAt',
       'ratingCreatedAt', 'helpfulnessLevel', 'fromNotification'],
      dtype='str')

In [16]:
df["ratingCreatedAt"].head()

0   2026-02-01 14:36:13.314
1   2026-02-01 14:36:04.345
2   2026-02-09 08:56:34.712
3   2026-02-09 00:51:34.502
4   2026-02-09 00:51:18.076
Name: ratingCreatedAt, dtype: datetime64[ms]

In [ ]:
# rating session

# order
df = df.sort_values(["raterParticipantId", "ratingCreatedAt"]).reset_index(drop=True)

# I want to arrange the rating times in a sequence, one before the other (+/-).
prev_time = df.groupby("raterParticipantId")["ratingCreatedAt"].shift(1)
next_time = df.groupby("raterParticipantId")["ratingCreatedAt"].shift(-1)

# calculate time difference
diff_prev = (df["ratingCreatedAt"] - prev_time).dt.total_seconds() / 60
diff_next = (next_time - df["ratingCreatedAt"]).dt.total_seconds() / 60

# Define session: within 5 minutes
df["is_rating_session"] = (
    (diff_prev <= 5) | (diff_next <= 5)
).fillna(False).astype(int)
df["is_rating_session"].mean()

np.float64(0.5904330012377618)

In [ ]:
# where the user rated a number of notes on the same post?

# Calculate the number of ratings for each (user, tweet).
cnt_user_post = (
    df.groupby(["raterParticipantId", "ratedOnTweetId"])
      .size()
      .reset_index(name="count_user_post")
)
cnt_user_post.head()

# Merge this count back into the original dataframe.
df = df.merge(
    cnt_user_post,
    on=["raterParticipantId", "ratedOnTweetId"],
    how="left"
)

# Define is_same_post_interest
df["is_same_post_interest"] = (
    df["count_user_post"] > 1
).astype(int)

df["is_same_post_interest"].mean()


np.float64(0.5092069016183216)

In [22]:
# Was the rating from a notification

df["is_notification"] = df["fromNotification"].astype(int)
df["is_notification"].mean()

np.float64(0.019806650681153636)

In [24]:
# rater swarm

# divide note group
df = df.sort_values(["noteId", "ratingCreatedAt"]).reset_index(drop=True)

# Calculate the total score for each note
note_total = (
    df.groupby("noteId")
      .size()
      .reset_index(name="note_total_count")
)

# Calculate the maximum number of ratings within a 1hour window.
import numpy as np

def max_count_in_one_hour(times):
    times = times.sort_values().tolist()
    max_count = 0
    
    for i in range(len(times)):
        start_time = times[i]
        count = 0
        
        for j in range(i, len(times)):
            if (times[j] - start_time).total_seconds() <= 3600:
                count += 1
            else:
                break
        
        if count > max_count:
            max_count = count
    
    return max_count

note_window = (
    df.groupby("noteId")["ratingCreatedAt"]
      .apply(max_count_in_one_hour)
      .reset_index(name="max_1h_count")
)

# merge 2 table and stat
note_stats = note_total.merge(
    note_window,
    on="noteId",
    how="left"
)

note_stats["is_swarm_note"] = (
    (note_stats["max_1h_count"] >= 10) &
    (note_stats["max_1h_count"] / note_stats["note_total_count"] > 0.5)
).astype(int)

note_stats["is_swarm_note"].mean()

df = df.merge(
    note_stats[["noteId", "is_swarm_note"]],
    on="noteId",
    how="left"
)

df["is_rater_swarm"] = df["is_swarm_note"]
df["is_rater_swarm"].mean()



np.float64(0.025856221157034204)

For this week and this issue, I analyze possible pathways through which users arrive at notes they rate. In addition, as Issue descirption said, for the first week calcaulate 4 part, 
1. Rating session (continuous block within 5 minutes) 
2. Same-post interest
3. Notification-based rating
4. Rater swarm behavior

For the first part, to determine whether a rating belongs to a "rating session," I interpreted the prompt as identifying moments when a user rated multiple notes within a consecutive time period. Since I couldn't observe explicit browsing sessions, I operationalized this concept by comparing the time intervals between consecutive ratings by the same user. I first sorted the data by ‘raterParticipantId’ and ‘ratingCreatedAt’, then used the groupby function, taking offsets of 1 and -1 respectively, to calculate the time difference between each rating and its preceding and following ratings. If any time interval was less than or equal to five minutes, the rating was marked as part of a session. I believe this method approximates the periods of active user engagement—times during which users are likely to browse and rate notes with focused attention.

For the second part, I interpreted the question as asking whether users were interested in a specific tweet, rather than a single comment. If a user rated multiple comments under the same ratedOnTweetId, it indicated that they were following the same post. To achieve this, I grouped the data by raterParticipantId and ratedOnTweetId, counted the number of ratings for each (user, tweet) combination, and then merged that count back into the main dataframe. Any rating with a count greater than 1 was marked as indicating interest in the same post.

For the third part, to determine whether a rating originated from a notification, I directly used the 'fromNotification' column from the dataset. Since this column already provides a boolean indicator, my implementation simply converted it to an integer binary variable (1 for true, 0 for false). This distinguishes between system-driven push notifications and user-driven explorations.

For the last one, I interpret this as identifying notes where the majority of ratings are concentrated within a one-hour period. To do this, I group the ratings by note ID, calculate the total number of ratings for each note, and then use a simple chronological loop to calculate the maximum number of ratings appearing within any one-hour window. If at least 10 ratings are concentrated within a one-hour period, and these ratings account for more than 50% of the total ratings for that note, then I mark that note as having a "rating cluster."

